In [1]:
import argparse
def args_parser():
    parser = argparse.ArgumentParser()
    # federated arguments
    parser.add_argument('--epochs', type=int, default=10, help="rounds of training")
    parser.add_argument('--num_users', type=int, default=100, help="number of users: K")
    parser.add_argument('--frac', type=float, default=0.1, help='the fraction of clients: C')
    parser.add_argument('--local_ep', type=int, default=5, help="the number of local epochs: E")
    parser.add_argument('--local_bs', type=int, default=10, help="local batch size: B")
    parser.add_argument('--lr', type=float, default=0.001, help='learning rate')
    parser.add_argument('--momentum', type=float, default=0.5, help='SGD momentum (default: 0.5)')

    # model arguments
    parser.add_argument('--model', type=str, default='mlp', help='model name')
    parser.add_argument('--kernel_num', type=int, default=9, help='number of each kind of kernel')
    parser.add_argument('--kernel_sizes', type=str, default='3,4,5',
                        help='comma-separated kernel size to use for convolution')
    parser.add_argument('--norm', type=str, default='batch_norm', help="batch_norm, layer_norm, or None")
    parser.add_argument('--num_filters', type=int, default=32,
                        help="number of filters for conv nets -- 32 for miniimagenet, 64 for omiglot.")
    parser.add_argument('--max_pool', type=str, default='True',
                        help="Whether use max pooling rather than strided convolutions")

    # other arguments
    parser.add_argument('--dataset', type=str, default='mnist', help="name of dataset")
    parser.add_argument('--iid', type=int, default=0, help='whether i.i.d or not, 1 for iid, 0 for non-iid')
    parser.add_argument('--num_classes', type=int, default=10, help="number of classes")
    parser.add_argument('--num_channels', type=int, default=3, help="number of channels of images")
    parser.add_argument('--gpu', type=int, default=-1, help="GPU ID")
    parser.add_argument('--stopping_rounds', type=int, default=10, help='rounds of early stopping')
    parser.add_argument('--verbose', type=int, default=1, help='verbose print, 1 for True, 0 for False')
    parser.add_argument('--seed', type=int, default=1, help='random seed (default: 1)')
    parser.add_argument('--ag_scalar', type=float, default=1.0, help="global aggregation updating scalar, simplicity for A Matrix")
    parser.add_argument('--lg_scalar', type=float, default=1.0, help="client local updating scalar, simplicity for S Matrix")
    parser.add_argument('--algorithm', type=str, default='fedavg', help='algorithm for optimization')

    args, _ = parser.parse_known_args()  # Extract only the args
    return args

args = args_parser()
print(f"Number of epochs: {args.epochs}")

Number of epochs: 10


In [2]:
import os
repo_dir = os.path.join(os.getcwd(), 'UnreliableClientsinFL')
os.chdir(repo_dir)
print("Working directory:", os.getcwd())

Working directory: C:\Users\osman\Documents\yl\UnreliableClientsinFL


In [3]:
# Düz eğitim
import matplotlib
import random

import matplotlib.pyplot as plt
import os
import copy
import numpy as np
from torchvision import datasets, transforms
import torch
import torch.nn.functional as F
from torch import autograd
from tensorboardX import SummaryWriter
from scipy import stats
import math

from sampling import mnist_iid, mnist_noniid, cifar_iid, cifar_noniid, server_test
#from options import args_parser
from Update import LocalFSVGRUpdate, LocalUpdate, ServerUpdate, LocalFedProxUpdate
from FedNets import MLP1, CNNMnist, CNNCifar
from averaging import average_weights



def test(net_g, data_loader, args):
    test_loss = 0
    correct = 0

    for idx, (data, target) in enumerate(data_loader):
        if args.gpu != -1:
            data, target = data.cuda(), target.cuda()
        data, target = autograd.Variable(data), autograd.Variable(target)
        log_probs = net_g(data)
        test_loss += F.nll_loss(log_probs, target, size_average=False).data[0]  # sum up batch loss
        y_pred = log_probs.data.max(1, keepdim=True)[1]  # get the index of the max log-probability
        correct += y_pred.eq(target.data.view_as(y_pred)).long().cpu().sum()

    test_loss /= len(data_loader.dataset)
    print('\nTest set: Average loss: {:.4f} \nAccuracy: {}/{} ({:.2f}%)\n'.format(
        test_loss, correct, len(data_loader.dataset),
        100. * correct / len(data_loader.dataset)))
    return correct, test_loss


def calculate_avg_grad(users_g):
    avg_grad = np.zeros(users_g[0][1].shape)
    total_size = np.sum([u[0] for u in users_g])  # Total number of samples of all users
    for i in range(len(users_g)):
        avg_grad = np.add(avg_grad,
                          users_g[i][0] * users_g[i][1])  # += users_g[i][0] * users_g[i][1]   <=> n_k * grad_k
    avg_grad = np.divide(avg_grad, total_size)  # /= total_size
    # print("avg_grad:", avg_grad)
    return avg_grad


def traverseList(nestList):
    flatList = []
    for item in nestList:
        if isinstance(item, list):
            flatList.extend(traverseList(item))
        else:
            flatList.append(item)
    return flatList


def flat(w):
    a = []
    for this_key in w.keys():
        a.append(list(copy.deepcopy(w[this_key].flatten().cpu().numpy())))
    return traverseList(a)


np.set_printoptions(threshold=100000)

if __name__ == '__main__':

    args = args_parser()
    algo_list = ['fedavg', 'fedprox', 'fsvgr']
    path_project = os.path.abspath('..')
    summary = SummaryWriter('local')
    args.gpu = 0  # -1 for CPU, 0 for first GPU
    args.lr = 0.002  # 0.001 for cifar dataset
    args.model = 'cnn'  # 'mlp' or 'cnn'
    args.dataset = 'mnist'  # 'cifar' or 'mnist'
    args.num_users = 8
    args.local_bs = 5
    args.probability = 1  # Probability of malicious user attack
    args.global_test = 2000

    args.malicious = True
    args.defense = False
    args.malicious_num = 1
    args.iter_num = 0
    args.q = 1
    args.iid = True
    args.verbose = False
    args.num_total_users = 8
    args.num_in_groups = 2
    args.malicious_num = [11, 12]

    print("dataset:", args.dataset, " num_users:", args.num_users, "local_bs:", args.local_bs)
    if args.malicious:
        print("malicious_num", args.malicious_num, "iter_num", args.iter_num, "probability", args.probability)


    def Function(local_ep, T):
        dict_users = {}
        dataset_train = []
        w_globals = []
        accuracy_list = []
        malicious = 0
        malicious_number = -1
        selected = 0
        s_selected = 1
        fine_tune = 0
        i = 1
        p = 0
        if args.dataset == 'mnist':
            dataset_train = datasets.FashionMNIST('../data/fashionmnist/', train=True, download=True,
                                                  transform=transforms.Compose([
                                                      transforms.ToTensor(),
                                                      transforms.Normalize((0.1307,), (0.3081,))
                                                  ]))
            dataset_test = datasets.FashionMNIST('../data/fashionmnist/', train=False, download=True,
                                                 transform=transforms.Compose([
                                                     transforms.ToTensor(),
                                                     transforms.Normalize((0.1307,), (0.3081,))
                                                 ]))
            # sample users
            if args.iid:
                dict_users = mnist_iid(dataset_train, args.num_users)
            else:
                dict_users = mnist_noniid(dataset_train, args.num_users)
        else:
            exit('Error: unrecognized dataset')

        img_size = dataset_train[0][0].shape
        net_glob = None

        if args.model == 'cnn' and args.dataset == 'mnist':
          if args.gpu != -1:
              torch.cuda.set_device(args.gpu)
              net_glob = CNNMnist(args=args).cuda()
              print(f"Using GPU: {torch.cuda.get_device_name(args.gpu)}")
          else:
              net_glob = CNNMnist(args=args)
        else:
            exit('Error: unrecognized model')
        print("Nerual Net:", net_glob)
        net_glob.train()  # Train() does not change the weight values
        w_glob = net_glob.state_dict()
        w_size = 0
        for k in w_glob.keys():
            size = w_glob[k].size()
            if (len(size) == 1):
                nelements = size[0]
            else:
                nelements = size[0] * size[1]
            w_size += nelements * 4
            # print("Size ", k, ": ",nelements*4)
        print("Weight Size:", w_size, " bytes")
        print("Weight & Grad Size:", w_size * 2, " bytes")
        print("Each user Training size:", 784 * 8 / 8 * args.local_bs, " bytes")
        print("Total Training size:", 784 * 8 / 8 * 60000, " bytes")

        dataset_c_test = server_test(dataset_test, args.global_test)
        net_start = copy.deepcopy(net_glob)

        epoch_acc = []
        epoch_loss = []
        server_acc = np.zeros(args.num_total_users + int(args.num_total_users/args.num_in_groups)*3, dtype='float')
        net_server = copy.deepcopy(net_start)
        net_glob = copy.deepcopy(net_start)
        dataset_users_test = server_test(dataset_test, 2000)
        global_test = copy.deepcopy(dataset_c_test)
        person = [[] for _ in range(args.num_users)]
        w_list = []
        glob_iter_num = int(math.ceil(T / local_ep)) + 0
        exit_condition = int(glob_iter_num + np.log2(args.num_total_users)*2)
        epochs = 216  # matches fed_learning.ipynb / 1_user_atmali.ipynb total round count
        print('epochs:', epochs, 'local_ep:', local_ep)
        com = 0
        y11 = []
        y12 = []
        w_all_locals, loss_all_locals, acc_all_locals = [], [], []
        for iter in range(epochs): # 0, 1, 2, 3, 4 ...
            print(f"iterations = {iter}")
            com += local_ep
            # if com > T:
            #     local_ep -= (com - T)
            #     com = 0
            epoch_server_acc = []
            print(f"Loacal_ep = {local_ep}")
            w_locals, loss_locals, acc_locals = [], [], []
            finish_condition2 = args.num_total_users

            for rounds in range(1, 1 + 1):
                for idx in range(1, finish_condition2 + 1):

                    client_selected = idx
                    local = LocalUpdate(args=args, dataset=dataset_train, idxs=dict_users[client_selected - 1], tb=summary)
                    w, loss, acc = local.update_weights(net=copy.deepcopy(net_glob), local_ep=int(local_ep))

                    for this_key in w.keys():
                          dev = np.random.normal(0, 0.03, w[this_key].size())
                          dev = torch.from_numpy(dev).float()
                          if args.gpu != -1:
                              dev = dev.cuda()
                          w[this_key] = w[this_key] + dev

                    w_locals.append(copy.deepcopy(w))
                    loss_locals.append(copy.deepcopy(loss))
                    acc_locals.append(copy.deepcopy(acc))
                w_all_locals.append(copy.deepcopy(w_locals))
                loss_all_locals.append(copy.deepcopy(loss_locals))
                acc_all_locals.append(copy.deepcopy(acc_locals))

            for idx in range(finish_condition2):
                net_server.load_state_dict(w_all_locals[rounds - 1][idx])
                server = ServerUpdate(args=args, dataset=dataset_test, idxs=dataset_users_test, tb=summary)
                acc, loss = server.test(net=net_server)
                epoch_server_acc.append(acc)
            weight = copy.deepcopy(epoch_server_acc)

            for i in range(len(weight)):
                weight[i] = 1
            total = sum(weight)
            for i in range(len(weight)):
                weight[i] /= total  # same weight average

            for i in range(len(w_locals)):
                w_list.append(copy.deepcopy(flat(w_locals[i])))

            w_glob = average_weights(w_locals, weight)
            w_globals.append(copy.deepcopy(w_glob))
            net_glob.load_state_dict(w_glob)
            net_local = ServerUpdate(args=args, dataset=dataset_test, idxs=global_test, tb=summary)
            return_acc, return_loss = net_local.test(net=net_glob)
            print("\nEpoch: {}, Global test loss {}, Global test acc: {:.2f}%".format(iter, return_loss,
                                                                                          100 * return_acc))

            if iter <= 200:
                if iter % 1 == 0 or iter == 0:
                    epoch_acc.append(return_acc)
                    epoch_loss.append(return_loss)
            else:
                if iter % 1 == 0:
                    epoch_acc.append(return_acc)
                    epoch_loss.append(return_loss)
        a = epoch_acc[-1]
        b = epoch_loss[-1]
        return a, b, y11, y12, epoch_acc, epoch_loss


    acc, loss, y1, y2, a, b = Function(8, 8*3)  # local_ep=8, matches fed_learning.ipynb cleanup-phase local_ep

dataset: mnist  num_users: 8 local_bs: 5
malicious_num [11, 12] iter_num 0 probability 1
NN: CNNMnist is created
Using GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Nerual Net: CNNMnist(
  (conv1): Conv2d(1, 10, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(10, 20, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=320, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=10, bias=True)
)
Weight Size: 67200  bytes
Weight & Grad Size: 134400  bytes
Each user Training size: 3920.0  bytes
Total Training size: 47040000.0  bytes
epochs: 216 local_ep: 8
iterations = 0
Loacal_ep = 8

Epoch: 0, Global test loss 1.3769469261169434, Global test acc: 52.25%
iterations = 1
Loacal_ep = 8

Epoch: 1, Global test loss 0.8792843818664551, Global test acc: 69.60%
iterations = 2
Loacal_ep = 8

Epoch: 2, Global test loss 0.8503217101097107, Global test acc: 69.25%
iterations = 3
Loacal_ep = 8

Epoch: 3, Global test loss 0.6982743740081787, Global test acc: 74.10%
iterat

In [4]:
# Save per-round global test accuracy history for cross-notebook comparison
import numpy as np
np.save("acc_history_clean_baseline.npy", np.array(a))
print(f"Saved {len(a)} rounds of accuracy history to acc_history_clean_baseline.npy")

Saved 216 rounds of accuracy history to acc_history_clean_baseline.npy
